# Nextflow on Verily Workbench

Run two RNA-seq examples on **Google Batch** from a Workbench JupyterLab cloud app.
Nextflow coordinates the workflow in the app and sends individual tasks to Batch.
Keep the app running until the workflow finishes.

This notebook teaches **direct `wb nextflow` execution**. For Workbench-managed
execution, start with [hello-nf-on-wb](hello-nf-on-wb/README.md) and the
[Workflows guide](https://support.workbench.verily.com/docs/guides/workflows/nextflow/).
Managed runs receive infrastructure configuration from Workbench; the configuration
below is for the direct CLI path. Direct runs do not become Workbench workflow jobs.

After shared setup, either example can be run independently. **Run All launches
both examples** and incurs compute and storage charges. Resume and destructive
cleanup are manual instructions, not executable cells.

## 1. Prerequisites and version baseline

Use a GCP-backed workspace and a JupyterLab cloud app with Python 3, Git, `wb`,
`gcloud`, and Nextflow installed locally. Authenticate `wb` and configure the
intended Workbench server/organization before continuing. Google Application
Default Credentials must also be valid for that workspace; a Workbench login
alone does not establish Google credentials. See [CLI basic usage](https://support.workbench.verily.com/docs/guides/cli/basic_usage/)
and the [Nextflow CLI guide](https://support.workbench.verily.com/docs/guides/cli/cli_nextflow/).

| Component | Tutorial baseline |
| --- | --- |
| Nextflow | 25.10.2, Groovy parser (`NXF_SYNTAX_PARSER=v1`) |
| `nextflow-io/rnaseq-nf` | Commit `bad0c709877f5091b27db3a7f38118424e242e3c`; bundled chicken input data; `nextflow/rnaseq-nf:v1.3.1` container |
| `nf-core/rnaseq` | Release 3.22.2 (requires Nextflow >=25.04.0); release-defined process containers |
| nf-core input data | `test-datasets` commit `626c8fab639062eade4b10747e919341cbf9b41a`; the test profile's yeast reference and sample sheet, with read URLs pinned below |

These versions are selected for this tutorial, not as the latest releases.
Pipeline source was checked; successful cloud execution still needs validation
in your workspace. Container tags and external services can change independently
of a source revision. The separate `nf-core` Python tool is not required.

Use existing writable bucket resources, or follow [workspace setup](workspace_description.md).
`ws_files` from [workspace_setup.ipynb](../workspace_setup.ipynb) is the default.
Store results in a bucket without automatic deletion. Scratch may expire, but
then it cannot be used to resume a run. No BigQuery dataset is needed.

### Settings

Fill in `WORKSPACE_ID`. Bucket settings are **resource IDs** in that workspace,
not GCS bucket names. Commands explicitly select that workspace without changing
your saved CLI selection. Choose a new `RUN_LABEL` for a new experiment; retain
it, the local launch directory, and cloud scratch when resuming an existing run.

In [ ]:
from pathlib import Path

WORKSPACE_ID = ""  # Workbench workspace ID, for example the ID shown by wb workspace list
SCRATCH_BUCKET_ID = "ws_files"
RESULTS_BUCKET_ID = "ws_files"
RUN_LABEL = "tutorial-01"
TUTORIAL_ROOT = Path.home() / "workbench-nextflow-tutorial"
NEXTFLOW_VERSION = "25.10.2"
RNASEQ_REVISION = "bad0c709877f5091b27db3a7f38118424e242e3c"
NFCORE_RELEASE = "3.22.2"
TEST_DATA_REVISION = "626c8fab639062eade4b10747e919341cbf9b41a"

### Command helpers

Every command checks its exit status. Failures raise an exception, stopping
Run All before dependent cells. The Workbench CLI passes tool arguments through
a shell, so the helper quotes each forwarded argument, including paths with spaces.
Only selected, non-secret context values are read; the notebook does not dump
your environment or credentials.

In [ ]:
import os
import re
import shlex
import shutil
import subprocess
import json

if not WORKSPACE_ID.strip():
    raise ValueError("Set WORKSPACE_ID in the settings cell before continuing.")
if not re.fullmatch(r"[A-Za-z0-9_-]+", RUN_LABEL):
    raise ValueError("RUN_LABEL must contain only letters, numbers, underscores or dashes.")
for tool in ("wb", "git", "gcloud", "nextflow"):
    if not shutil.which(tool):
        raise RuntimeError(f"Install {tool} in this cloud app and add it to PATH first.")

LAUNCH_ROOT = TUTORIAL_ROOT / RUN_LABEL
LAUNCH_ROOT.mkdir(parents=True, exist_ok=True)
TOOL_ENV = dict(os.environ, NXF_VER=NEXTFLOW_VERSION, NXF_SYNTAX_PARSER="v1")

def run(args, *, cwd=LAUNCH_ROOT, capture=False):
    """Run an argv list, stream output, and fail on a nonzero exit status."""
    args = [str(arg) for arg in args]
    print("$", shlex.join(args))
    if capture:
        result = subprocess.run(args, cwd=cwd, env=TOOL_ENV, text=True,
                                stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        if result.stderr:
            print(result.stderr, end="")
        result.check_returncode()
        return result.stdout.strip()
    with subprocess.Popen(args, cwd=cwd, env=TOOL_ENV, text=True,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
            raise
        if returncode:
            raise subprocess.CalledProcessError(returncode, args)

def wb_tool(tool, *args, **kwargs):
    return run(["wb", tool, f"--workspace={WORKSPACE_ID}",
                *[shlex.quote(str(arg)) for arg in args]], **kwargs)

def resolve_bucket(resource_id):
    uri = run(["wb", "resource", "resolve", f"--workspace={WORKSPACE_ID}",
               f"--id={resource_id}", "--format=TEXT"], capture=True)
    if not re.fullmatch(r"gs://[a-z0-9][a-z0-9._-]+", uri):
        raise ValueError(f"{resource_id!r} did not resolve to a GCS bucket: {uri!r}")
    return uri

In [ ]:
run(["wb", "version"])
run(["wb", "auth", "status"])
run(["wb", "workspace", "describe", f"--workspace={WORKSPACE_ID}", "--format=TEXT"])
mode = run(["wb", "config", "get", "utility-mode", "--format=TEXT"], capture=True)
if mode != "LOCAL_PROCESS":
    raise RuntimeError("This notebook requires local tools in a cloud app. "
                       "Configure 'wb config set utility-mode LOCAL_PROCESS' in a terminal, "
                       "then rerun this cell. This changes your CLI configuration.")
version_text = wb_tool("nextflow", "-version", capture=True)
print(version_text)
if not re.search(r"version\s+" + re.escape(NEXTFLOW_VERSION) + r"\b", version_text):
    raise RuntimeError(f"Expected Nextflow {NEXTFLOW_VERSION}; check the local installation.")

### Resolve the workspace context

The CLI injects workspace context into its child processes. Variables in the
notebook kernel or an ordinary app terminal may be missing or refer to another
context. Read the project, service account and workspace region through `wb`.
Use the workspace region for `google.location` and its regional subnetwork.
A metadata-server lookup of the app VM's zone is not needed.

In [ ]:
# Refresh the resource cache before resolving bucket IDs.
run(["wb", "resource", "list", f"--workspace={WORKSPACE_ID}", "--format=TEXT"])
context_keys = ["GOOGLE_CLOUD_PROJECT", "GOOGLE_SERVICE_ACCOUNT_EMAIL", "PROJECT_DEFAULT_REGION"]
context_values = run(["wb", "utility", "execute", f"--workspace={WORKSPACE_ID}",
                      "--", "printenv", *context_keys], capture=True).splitlines()
if len(context_values) != len(context_keys) or not all(context_values):
    raise RuntimeError("Workbench did not supply a complete GCP context. Check the workspace and CLI version.")
PROJECT, SERVICE_ACCOUNT, REGION = context_values
if not re.fullmatch(r"[a-z]+-[a-z]+[0-9]+", REGION):
    raise ValueError(f"Expected a workspace region (not a zone), got {REGION!r}.")
SCRATCH_BUCKET = resolve_bucket(SCRATCH_BUCKET_ID)
RESULTS_BUCKET = resolve_bucket(RESULTS_BUCKET_ID)
print(json.dumps({"workspace": WORKSPACE_ID, "project": PROJECT, "region": REGION,
                  "service_account": SERVICE_ACCOUNT, "scratch": SCRATCH_BUCKET,
                  "results": RESULTS_BUCKET, "launch_directory": str(LAUNCH_ROOT)}, indent=2))

### Shared Batch configuration and source checkout

The helper writes a separate `workbench.config` inside each example's launch
directory. It does not edit the pipeline or your home-directory Nextflow config.
`-C` explicitly selects the pipeline config and this override, excluding unrelated
home/launch configs. The custom `workbench_batch` profile avoids reusing an
upstream profile with different network or data defaults.

Nextflow's `config -flat` command checks configuration expansion, **not** Batch
permissions, scheduling or container access. Each run below uses the same config
and profile selection. Each example has its own scratch path and durable outdir.

In [ ]:
def checkout(repository, revision, destination):
    """Create an isolated checkout, or verify an unchanged existing one."""
    if not destination.exists():
        run(["git", "clone", "--no-checkout", repository, destination])
        run(["git", "-C", destination, "checkout", "--detach", revision])
    origin = run(["git", "-C", destination, "remote", "get-url", "origin"], capture=True)
    head = run(["git", "-C", destination, "rev-parse", "HEAD"], capture=True)
    expected = run(["git", "-C", destination, "rev-parse", f"{revision}^{{commit}}"], capture=True)
    dirty = run(["git", "-C", destination, "status", "--porcelain"], capture=True)
    if origin != repository or head != expected or dirty:
        raise RuntimeError(f"Existing checkout {destination} differs from the requested source. "
                           "Choose a fresh TUTORIAL_ROOT; do not overwrite your changes.")
    return head

def groovy_string(value):
    return "'" + str(value).replace("\\", "\\\\").replace("'", "\\'") + "'"

def prepare_example(name, pipeline, profiles, *, container=None):
    launch = LAUNCH_ROOT / name
    launch.mkdir(exist_ok=True)
    scratch = f"{SCRATCH_BUCKET}/nextflow-tutorial/{RUN_LABEL}/{name}/work"
    outdir = f"{RESULTS_BUCKET}/nextflow-tutorial/{RUN_LABEL}/{name}/results"
    settings = {
        "process.executor": "google-batch",
        "google.project": PROJECT,
        "google.location": REGION,
        "google.batch.serviceAccountEmail": SERVICE_ACCOUNT,
        "google.batch.network": "global/networks/network",
        "google.batch.subnetwork": f"regions/{REGION}/subnetworks/subnetwork",
        "workDir": scratch,
    }
    if container:
        settings["process.container"] = container
    lines = ["profiles {", "    workbench_batch {"]
    lines.extend(f"        {key} = {groovy_string(value)}" for key, value in settings.items())
    lines += ["        google.batch.usePrivateAddress = true", "    }", "}", ""]
    config = launch / "workbench.config"
    config.write_text("\n".join(lines))
    return {"name": name, "pipeline": pipeline, "launch": launch,
            "config_files": f"{pipeline / 'nextflow.config'},{config}",
            "profiles": ",".join([*profiles, "workbench_batch"]),
            "scratch": scratch, "outdir": outdir, "expected_config": settings}

def inspect_config(example):
    expanded = wb_tool("nextflow", "-C", example["config_files"], "config",
                       example["pipeline"], "-profile", example["profiles"], "-flat",
                       cwd=example["launch"], capture=True)
    (example["launch"] / "expanded.config").write_text(expanded + "\n")
    values = {}
    for line in expanded.splitlines():
        key, separator, value = line.partition(" = ")
        if separator:
            values[key.strip()] = value.strip().strip("'\"")
    for key, expected in example["expected_config"].items():
        if values.get(key) != expected:
            raise RuntimeError(f"Unexpected {key}: {values.get(key)!r}; expected {expected!r}.")
    if values.get("google.batch.usePrivateAddress") != "true":
        raise RuntimeError("Expected private Batch VM addresses.")
    print("Batch configuration checked. Full config:", example["launch"] / "expanded.config")
    print("Scratch:", example["scratch"], "\nResults:", example["outdir"])

def launch_example(example, pipeline_args=(), *, resume=None):
    args = ["-C", example["config_files"], "run", str(example["pipeline"] / "main.nf"),
            "-profile", example["profiles"], "-ansi-log", "false",
            "--outdir", example["outdir"], *pipeline_args]
    if resume:
        args += ["-resume", resume]
    # Use distinct local report names when rerunning/resuming in the same directory.
    from datetime import datetime, timezone
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    args += ["-with-report", f"execution-{stamp}.html", "-with-trace", f"trace-{stamp}.tsv"]
    (example["launch"] / f"command-{stamp}.json").write_text(json.dumps(args, indent=2) + "\n")
    wb_tool("nextflow", *args, cwd=example["launch"])
    print("Nextflow exited successfully. Verify the published report in the next cell.")

def fetch_report(example, relative_path):
    source = f"{example['outdir']}/{relative_path}"
    destination = example["launch"] / "multiqc_report.html"
    wb_tool("gcloud", "storage", "cp", source, destination)
    if not destination.exists() or destination.stat().st_size == 0:
        raise RuntimeError(f"Expected a nonempty report: {destination}")
    print("Downloaded report:", destination)
    return destination

## 2. Example 1: a small RNA-seq pipeline

[`nextflow-io/rnaseq-nf`](https://github.com/nextflow-io/rnaseq-nf/tree/bad0c709877f5091b27db3a7f38118424e242e3c)
quantifies a small **chicken gut** dataset with Salmon and produces FastQC and
MultiQC reports. The reads, reference and MultiQC configuration are bundled in
the pinned repository; this example does not depend on an external input bucket.
The source revision uses the `--outdir` parameter. Changing to a newer pipeline
revision may change that interface, so update the commands and checks together.

In [ ]:
rnaseq_source = TUTORIAL_ROOT / f"rnaseq-nf-{RNASEQ_REVISION[:12]}"
rnaseq_commit = checkout("https://github.com/nextflow-io/rnaseq-nf.git", RNASEQ_REVISION, rnaseq_source)
rnaseq = prepare_example("rnaseq-nf", rnaseq_source, ["standard"],
                         container="docker.io/nextflow/rnaseq-nf:v1.3.1")
inspect_config(rnaseq)

### Launch and verify

This cell runs cloud tasks and waits for Nextflow to finish. Do not stop the
cloud app while it is running. An exception indicates a failed command; inspect
its output and `.nextflow.log` before proceeding.

In [ ]:
launch_example(rnaseq)

In [ ]:
rnaseq_report = fetch_report(rnaseq, "multiqc_report.html")
wb_tool("nextflow", "log", cwd=rnaseq["launch"])

## 3. Example 2: nf-core RNA-seq

[`nf-core/rnaseq` 3.22.2](https://nf-co.re/rnaseq/3.22.2/) provides a larger RNA-seq
workflow. Its `test` profile uses small yeast reference files and caps requested
resources for testing. These are demonstration inputs, not a full biological analysis.
The `docker` profile selects OCI containers; **Google Batch**, not Docker on the
app, runs the tasks. Keep the pipeline's process-specific images: replacing all
of them with the teaching image from Example 1 would remove required software.

The release's [test profile](https://github.com/nf-core/rnaseq/blob/3.22.2/conf/test.config)
pins the reference and sample-sheet revision, but URLs *inside* the sheet still
use a moving branch. The following cell pins those read URLs to the same commit
and saves a local copy. No separate test-data repository resource is needed.

In [ ]:
import csv
import io
from urllib.request import urlopen

nfcore_source = TUTORIAL_ROOT / f"nf-core-rnaseq-{NFCORE_RELEASE}"
nfcore_commit = checkout("https://github.com/nf-core/rnaseq.git", NFCORE_RELEASE, nfcore_source)
nfcore = prepare_example("nf-core-rnaseq", nfcore_source, ["test", "docker"])
data_base = f"https://raw.githubusercontent.com/nf-core/test-datasets/{TEST_DATA_REVISION}/"
sheet_url = data_base + "samplesheet/v3.10/samplesheet_test.csv"
with urlopen(sheet_url, timeout=60) as response:
    reader = csv.DictReader(io.StringIO(response.read().decode("utf-8")))
    fieldnames = reader.fieldnames
    rows = list(reader)
if fieldnames != ["sample", "fastq_1", "fastq_2", "strandedness"] or not rows:
    raise ValueError("Unexpected test sample-sheet format.")
branch_base = "https://raw.githubusercontent.com/nf-core/test-datasets/rnaseq/"
for row in rows:
    for column in ("fastq_1", "fastq_2"):
        url = row[column]
        if url.startswith(branch_base):
            url = data_base + url[len(branch_base):]
        if url and not url.startswith(data_base):
            raise ValueError(f"Unexpected read URL: {url}")
        row[column] = url
samplesheet = nfcore["launch"] / "samplesheet.csv"
with samplesheet.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)
print(samplesheet.read_text())
inspect_config(nfcore)

### Launch and verify

This example uses the default `star_salmon` aligner. BBSplit is skipped to keep
the tutorial focused on RNA-seq and avoid an extra reference-manifest download.
The test profile supplies the small reference; do not add the old `--genome
R64-1-1` option, which would select a different reference setup.

In [ ]:
nfcore_args = ["--input", str(samplesheet), "--aligner", "star_salmon", "--skip_bbsplit", "true"]
launch_example(nfcore, nfcore_args)

In [ ]:
nfcore_report = fetch_report(nfcore, "multiqc/star_salmon/multiqc_report.html")
wb_tool("nextflow", "log", cwd=nfcore["launch"])

## 4. Reports, monitoring and troubleshooting

Open the downloaded `multiqc_report.html` in your browser. If JupyterLab's HTML
viewer disables JavaScript, download the report using the file browser and open
it locally. Inspect quality metrics as well as the pipeline exit status.

For **direct CLI runs**, use the example's local launch directory:

- `.nextflow.log` records orchestration and task failures.
- `execution-*.html` and `trace-*.tsv` record execution details.
- `wb nextflow --workspace=<workspace-id> log` lists local run history; run it
  from the same example directory to see the relevant sessions.
- `wb nextflow --workspace=<workspace-id> log <run-name> -f
  'task_id,name,status,duration,cpus,container'` shows task details.
- In the [Google Batch console](https://console.cloud.google.com/batch/jobs),
  select the printed workspace project and region, open a task job, and inspect
  **Events** and logs. You can also use `wb gcloud --workspace=<workspace-id>
  batch jobs list --location=<region>`.

For **managed Workflows**, use the Workbench job and task views instead. The
local run history in this notebook does not list those managed runs.

Common failures:

| Symptom | What to check |
| --- | --- |
| Authentication failure | Workbench login and Google Application Default Credentials are separate. Follow your cloud app's authentication instructions; restarting the kernel does not refresh credentials. |
| Bucket resolution fails | Check the workspace and resource ID, then refresh with `wb resource list --workspace=<workspace-id>`. |
| Region/subnet error | Match `google.location` and `regions/<region>/subnetworks/subnetwork` to the workspace region. |
| Tasks cannot pull images | Check Batch events for registry access, rate limits and private-network egress. Follow the Workbench guide for Artifact Registry mirrors. For nf-core, preserve each process's image and tag when mirroring. |
| nf-core setup fails before submission | The app needs access to GitHub, Nextflow plugins, and upstream shared configuration. Record the error and actual versions; do not disable validation to hide it. |
| App stopped or kernel interrupted | Inspect Nextflow and Batch status before relaunching. Interrupting the notebook is not proof that all cloud tasks were cancelled. |
| Resume reruns tasks | Both the local `.nextflow` cache and cloud work files are required, and inputs/configuration must match. Expired scratch cannot be reused. |

The [Workbench CLI guide](https://support.workbench.verily.com/docs/guides/cli/cli_nextflow/)
contains additional Batch troubleshooting steps.

## 5. Optional: resume a direct CLI run

Use **`-resume`** (one dash), followed by the session ID from `nextflow log`.
Keep the same pipeline revision, settings, local launch directory and scratch
path. Published results alone are insufficient; keep the local `.nextflow` cache
and Batch work files. Do not change the outdir merely to resume.

After running the shared setup and the relevant example's preparation cells,
copy ONE of these into a new code cell, replacing the session ID. These are
Markdown examples so Run All does not submit extra jobs:

```python
# Example 1
launch_example(rnaseq, resume="YOUR_SESSION_ID")
fetch_report(rnaseq, "multiqc_report.html")
```

```python
# Example 2
nfcore_args = ["--input", str(samplesheet), "--aligner", "star_salmon", "--skip_bbsplit", "true"]
launch_example(nfcore, nfcore_args, resume="YOUR_SESSION_ID")
fetch_report(nfcore, "multiqc/star_salmon/multiqc_report.html")
```

These instructions concern direct CLI runs, not managed Workflows resume support.
See [Nextflow caching and resume](https://www.nextflow.io/docs/stable/cache-and-resume.html).

## 6. Optional: clean scratch manually

Cleanup removes work files and cached state needed for resume. First verify a
successful run and its durable result files. Select a **single run name** from
that example's local history. From a terminal in its launch directory, preview:

```sh
wb nextflow --workspace=<workspace-id> clean -n <run-name>
```

Review the exact paths. Only if you intend to delete those files, run:

```sh
wb nextflow --workspace=<workspace-id> clean -f <run-name>
```

There is no executable deletion cell and no “all runs” shortcut. Do not delete
the result bucket or use this local-history procedure for managed Workflows.

## 7. Record provenance

Save the engine, pipeline revisions and paths for whichever examples you ran.
This records configuration and source provenance, not a claim of successful
cloud execution. Keep command JSON, trace and reports with your results.

In [ ]:
from datetime import datetime, timezone

provenance = {"recorded_at": datetime.now(timezone.utc).isoformat(),
              "nextflow": NEXTFLOW_VERSION, "workspace": WORKSPACE_ID,
              "project": PROJECT, "region": REGION,
              "run_label": RUN_LABEL, "test_data_revision": TEST_DATA_REVISION,
              "pipelines": {}}
for variable, revision_variable in (("rnaseq", "rnaseq_commit"), ("nfcore", "nfcore_commit")):
    if variable in globals():
        example = globals()[variable]
        provenance["pipelines"][variable] = {"commit": globals()[revision_variable],
                                "profiles": example["profiles"],
                                "scratch": example["scratch"], "outdir": example["outdir"]}
provenance_path = LAUNCH_ROOT / "provenance.json"
provenance_path.write_text(json.dumps(provenance, indent=2) + "\n")
print(provenance_path)

---
Copyright 2022 Verily Life Sciences LLC

Use of this source code is governed by the BSD-style license in [LICENSE](../LICENSE).